In [1]:
# Core imports
import os
import sys
from pathlib import Path
from datetime import datetime, date
from dotenv import load_dotenv
import pandas as pd
from edgar import Company, set_identity
from typing import Optional, Literal
import asyncio

# Load environment variables
load_dotenv()

# Set SEC identity
sec_identity = os.getenv("SEC_ID")
if not sec_identity:
    raise ValueError("SEC_ID not found in .env file")
set_identity(sec_identity)

print("✓ Core imports loaded")
print(f"✓ SEC identity configured: {sec_identity.split()[0]}")

✓ Core imports loaded
✓ SEC identity configured: pedroemail@duck.com


/home/pedro/projects/fin_import2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import asyncio
from extractors.income_statement_extractor import get_filing, extract_income_statement

async def test():
    print("Testing extraction with Ollama AI fallback...\n")
    
    # Get AAPL 2024 filing
    filing = get_filing('AAPL', '10-K', 2024)
    
    # Extract with AI enabled
    result = await extract_income_statement(
        filing, 
        'AAPL', 
        '10-K', 
        2024, 
        use_ai_fallback=True
    )
    
    # Show results
    print("\n" + "="*80)
    print("RESULTS")
    print("="*80)
    
    found = len(result[result['Status'] == '✓'])
    total = len(result)
    coverage = (found / total) * 100
    
    print(f"\nCoverage: {coverage:.1f}% ({found}/{total} fields)")
    print(f"\nFields found:")
    print(result[result['Status'] == '✓'][['Field', 'Value']].to_string(index=False))

await test()

In [ ]:
#### BULK IMPORT 10Ks ####
import asyncio
from bulk_import_10k import bulk_import_10k

# Run bulk import
results = await bulk_import_10k(
    ticker_csv='test_tickers.csv',
    periods=5,                              # Last 10 years
    db_path='data/financial_statements.duckdb',
    use_ai_fallback=True,                   # Set True for better coverage
    skip_existing=True,                      # Skip already-imported filings
    rate_limit_delay=1.0                     # 1 second between requests
)

✓ Loaded comprehensive XBRL mapping (125 concepts, 30 fields)
✓ Loaded balance sheet XBRL mapping (62 concepts, 37 fields)
✓ Loaded cash flow XBRL mapping (50 concepts, 31 fields)
BULK 10-K IMPORT

📋 Reading tickers from test_tickers.csv...
✓ Found 1 tickers

💾 Connecting to database: data/financial_statements.duckdb...
✓ Database schema created/verified
✓ Connected to financial statements database: data/financial_statements.duckdb

🚀 Starting bulk extraction...
   Rate limit: 1.0s between requests
   AI fallback: Enabled
   Skip existing: Yes


[1/1] Processing DE...
--------------------------------------------------------------------------------

EXTRACTING INCOME STATEMENT

✓ Available periods: 2022-10-30, 2021-10-31, 2020-11-01
✓ Using most recent period: 2022-10-30
✓ AI fallback enabled (batch mode)

Extracting 30 line items...

  Pass 1 (static): 16/30 fields found
  Pass 2 (AI): resolving 14 unfound fields...
  Pass 2: 14 unfound fields, 1 unmapped concepts
  DB lookup: found 17

In [ ]:

from dotenv import load_dotenv
from agents import Agent, Runner, OpenAIChatCompletionsModel, ModelSettings, RunConfig
#from agents.mcp import MCPServerStdio ## NEEDED ONLY WITH MCP
from openai import AsyncOpenAI
# Load environment variables
load_dotenv(override=True)


openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
if not openrouter_api_key:
    raise ValueError("OPENROUTER_API_KEY not found in environment variables")


openrouter_extra_body={
    "provider": {
        "only": ["cerebras"],        # restrict to Cerebras
        "allow_fallbacks": False,    # fail instead of switching providers
    },
}    

openrouter_client = AsyncOpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)
openrouter_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=openrouter_client)


openrouter_agent = Agent(
    name="Agent Joker",
    instructions="Tell a joke",
    model=openrouter_model,
    model_settings=ModelSettings(extra_body=openrouter_extra_body),
)
result = await Runner.run(openrouter_agent, input="Tell a joke about a chicken", run_config=RunConfig(tracing_disabled=True),)

print(result.final_output)